# Tang Nano 20K - UART Echo Test (uart3.sv)

**Hardware**: Tang Nano 20K (27MHz) com CH340 USB-Serial  
**Baud Rate**: 115200 bps  
**Funcionalidade**: RX → TX (retransmissão automática)

**Pinagem**:
- `clk` = A9 (27 MHz)
- `btn1` = C11 (Reset)
- `uart_rx` = B10
- `uart_tx` = C10

In [1]:
import serial
import time

In [5]:
class UARTEchoTester:
    def __init__(self, port='/dev/ttyUSB0', baudrate=115200):
        self.port = port
        self.baudrate = baudrate
        self.ser = None
    
    def connect(self):
        try:
            self.ser = serial.Serial(self.port, self.baudrate, timeout=1.0)
            time.sleep(0.5)
            print(f"Conectado: {self.port} @ {self.baudrate} bps")
            return True
        except Exception as e:
            print(f"Erro: {e}")
            return False
    
    def disconnect(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
            print("Desconectado")
    
    def send_receive(self, message, timeout=2.0):
        """Envia mensagem e aguarda echo"""
        if not self.ser:
            print("Não conectado")
            return None
        
        self.ser.write(message)
        start = time.time()
        response = b''
        
        while (time.time() - start) < timeout:
            if self.ser.in_waiting > 0:
                response += self.ser.read(self.ser.in_waiting)
                if len(response) >= len(message):
                    break
            time.sleep(0.01)
        
        return response
    
    def test_single_byte(self, byte_val=0x41):  # 'A'
        """Testa envio de um byte único"""
        print(f"\nTeste Single Byte (0x{byte_val:02x} = '{chr(byte_val)}')")
        response = self.send_receive(bytes([byte_val]), timeout=1.0)
        
        if response and response[0] == byte_val:
            print(f"SUCESSO: Enviou {hex(byte_val)} e recebeu {hex(response[0])}")
            return True
        else:
            print(f"FALHA: Esperado {hex(byte_val)}, recebido {response.hex() if response else 'nada'}")
            return False
    
    def test_string(self, text="Hello"):
        """Testa envio de string"""
        print(f"\nTeste String: '{text}'")
        response = self.send_receive(text.encode(), timeout=2.0)
        
        if response:
            response_str = response.decode('ascii', errors='replace')
            print(f"Enviado: {repr(text)}")
            printsx(f"Recebido: {repr(response_str)}")
            
            if response == text.encode():
                print(f"SUCESSO: Echo perfeito!")
                return True
            else:
                print(f"Recebido mas com diferenças")
                return False
        else:
            print(f"FALHA: Nenhum dado recebido")
            return False
    
    def test_rapid_bytes(self, count=10):
        """Testa envio rápido de múltiplos bytes"""
        print(f"\nTeste Rápido ({count} bytes)")
        test_data = bytes(range(48, 48 + count))  # 0x30-0x39 ('0'-'9')
        response = self.send_receive(test_data, timeout=3.0)
        
        if response and response == test_data:
            print(f"SUCESSO: Todos os {count} bytes ecoados corretamente")
            return True
        else:
            print(f"FALHA: Esperado {test_data.hex()}")
            print(f"         Recebido {response.hex() if response else 'nada'}")
            return False

In [4]:
# Criar testador
tester = UARTEchoTester(port='/dev/ttyUSB1')

tester.connect()

tester.test_single_byte(0x41)  # 'A'

✅ Conectado: /dev/ttyUSB1 @ 115200 bps

🧪 Teste Single Byte (0x41 = 'A')
❌ FALHA: Esperado 0x41, recebido nada


False

In [10]:
tester.connect()

✅ Conectado: /dev/ttyUSB0 @ 115200 bps


True

In [14]:
tester.test_single_byte(0x41)  # 'A'


🧪 Teste Single Byte (0x41 = 'A')
❌ FALHA: Esperado 0x41, recebido fa41


False

## Testes Disponíveis

## Status dos Testes

✅ = Teste passou  
❌ = Teste falhou  
⚠️ = Teste parcial

In [4]:
tester.test_string("Hello")


🧪 Teste String: 'Hello'
❌ FALHA: Nenhum dado recebido


False

In [15]:
tester.test_rapid_bytes(10)


🧪 Teste Rápido (10 bytes)
❌ FALHA: Esperado 30313233343536373839
         Recebido fa30ffffffffffff


False

In [6]:
tester.disconnect()

✅ Desconectado


## Build & Program

```bash
cd /home/anderson/github_projects/embedded_systems/hdl/tang_nano_20k/uart_simples

# Compile with Gowin EDA or oss-cad-suite
# Result: uart3.fs (bitstream)

# Program Tang Nano with gowin-programmer or IDE
```

After programming, run tests above.